# Explainable Retrieval

Show *why* each passage was retrieved and cite sources in the answer.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def explain(query, chunk):
    return llm(f"In one sentence, explain how this passage relates to the question.\n\nQuestion: {query}\n\nPassage: {chunk}")

query = "How do I stop a model from making things up?"
hits = store.search(query, 3)
for n, h in enumerate(hits, 1):
    print(f"[{n}] similarity={h['score']:.3f}")
    print("    why:", explain(query, h["text"]))
    print("    text:", h["text"][:120], "...\n")

Ask for inline `[n]` citations so every claim can be traced to a source chunk.

In [ ]:
numbered = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
print(llm(
    f"Context:\n{numbered}\n\nQuestion: {query}\n\nAnswer using only the context. After each claim add the source number like [1].",
    system=RAG_SYSTEM,
))